In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import numpy as np

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts.rq2_function_lib import set_plot_style
from scripts.rq9_function_lib import display_weather_per_region, display_weather_codes_per_region, levene_test_for_extreme_weather 
from scripts.rq9_function_lib import display_levene_test_results, extract_arrays_for_global_test, calculate_global_levene

set_plot_style()

# What impact do extreme weather conditions have on the fuel prices?

In [2]:
weather_path = Path(r'/Users/sebastian/data-science-projekt/weather_per_leitregion')
region_price_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/regions_avg_prices_per_year')

In [3]:
display_weather_per_region(weather_path, "24", "2020")

Reading file from: /Users/sebastian/data-science-projekt/weather_per_leitregion/weather_region24.csv
df successfully created.
Downsampling df...


Display weather codes

In [4]:
display_weather_codes_per_region(weather_path, "24", "2018")

Reading file from: /Users/sebastian/data-science-projekt/weather_per_leitregion/weather_region24.csv
df successfully created.
Downsampling df...


Now we want to check with a Levene test, if an extreme weather event influences the fuel prices. For that we calculated the median price for each german 'Postleitzahl-Leitregion' and collected the historical weather data for the geographical middle of each region.

Weiter und genauer erklären!!!!

In [5]:
regions_df = pd.read_csv(Path(r'/Users/sebastian/data-science-projekt/plz_leitregionen.csv'),dtype={"leit_plz":str}) #change if necessary
regions = regions_df["leit_plz"].to_list()


In [6]:
all_regional_results = []
global_arrays = {}

for region in regions:
    # do regional test first
    result_list = levene_test_for_extreme_weather(weather_path, region_price_path, region)
    all_regional_results.extend(result_list)

    # collect arrays for gloabl test
    arrays_global_levene = extract_arrays_for_global_test(weather_path, region_price_path, region)

    if arrays_global_levene["status"] == "Success":
        for var_name, arrays in arrays_global_levene["arrays"].items():
            if var_name not in global_arrays:
                global_arrays[var_name] = {"event": [], "control": []}
            
            global_arrays[var_name]["event"].append(arrays["event"])
            global_arrays[var_name]["control"].append(arrays["control"])

print("Data collected. Calculating global levene...")
global_results = calculate_global_levene(global_arrays)

#convert to pandas df
global_result_df = pd.DataFrame(global_results)
result_df = pd.DataFrame(all_regional_results)


Data collected. Calculating global levene...


Regional results of the Levene test.

In [7]:
display_levene_test_results(result_df)

,region,variable,test_statistic,p_value,significant,data_event,data_controll,note
0,01,diesel_median,15.71,0.0001,True,60894.000000,12328.000000,Success!
1,01,e5_median,0.12,0.7341,False,60894.000000,12328.000000,Success!
2,01,e10_median,4.18,0.0408,True,60894.000000,12328.000000,Success!
3,02,diesel_median,1.69,0.1941,False,60075.000000,10319.000000,Success!
4,02,e5_median,0.00,0.9815,False,60075.000000,10319.000000,Success!
5,02,e10_median,7.48,0.0062,True,60075.000000,10319.000000,Success!
6,03,diesel_median,27.68,0.0000,True,52909.000000,14273.000000,Success!
7,03,e5_median,98.16,0.0000,True,52909.000000,14273.000000,Success!
8,03,e10_median,114.42,0.0000,True,52909.000000,14273.000000,Success!
9,04,diesel_median,1.21,0.2707,False,61561.000000,13460.000000,Success!


Result for the Levene test calculated over all of Germany

TODO: texte ordentlich & Methoden Kommentare

In [8]:
display_levene_test_results(global_result_df)

,variable,test_statistic,p_value,significant,data_event,data_controll,note
0,diesel_median,4.59,0.0322,True,5922708,1093198,Global Success!
1,e5_median,6.66,0.0098,True,5922708,1093198,Global Success!
2,e10_median,11.73,0.0006,True,5922708,1093198,Global Success!
